# Build the Favorita uncleaned merged base

This notebook builds the first real-data merged training base for EDIP from the official Favorita raw CSV files. It performs structural joins and validation only.

**Execution boundary:** `train.csv` remains the base table and is processed in bounded chunks. The notebook does not use `test.csv`, synthetic EDIP data, database extracts, old feature artifacts, or prior model outputs. It performs no cleaning, imputation of sales values, feature engineering, splitting, or modelling.

## 1. Purpose and scope

The required final grain is one row per `(date, store_nbr, item_nbr)`. Every reference join is validated as many-to-one, and the original training row count must be preserved. Missing reference values remain visible for later cleaning decisions, except that absence from the store-date holiday lookup is explicitly represented as `False`/`0` for holiday flags and event count.

## 2. Imports and configuration

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import math
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import display

TRAIN_CHUNKSIZE = 1_000_000
PARQUET_COMPRESSION = "zstd"
PARQUET_ROW_GROUP_SIZE = 250_000
GRAIN_COLUMNS = ["date", "store_nbr", "item_nbr"]

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the EDIP repository.")

REPO_ROOT = find_repo_root(Path.cwd())
RAW_DIR = REPO_ROOT / "data" / "raw" / "favorita-grocery-sales-forecasting"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
MERGED_DIR = PROCESSED_DIR / "favorita_merged"
CLEANED_DIR = PROCESSED_DIR / "favorita_cleaned"
FEATURES_DIR = PROCESSED_DIR / "features"
OUTPUT_PATH = MERGED_DIR / "favorita_merged_base.parquet"
TEMP_OUTPUT_PATH = MERGED_DIR / "favorita_merged_base.parquet.inprogress"

for directory in (MERGED_DIR, CLEANED_DIR, FEATURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print({
    "train_chunksize": TRAIN_CHUNKSIZE,
    "parquet_compression": PARQUET_COMPRESSION,
    "parquet_row_group_size": PARQUET_ROW_GROUP_SIZE,
    "output_path": OUTPUT_PATH.relative_to(REPO_ROOT).as_posix(),
    "storage_approach": "single Parquet file written incrementally in bounded row groups",
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
})

{'train_chunksize': 1000000, 'parquet_compression': 'zstd', 'parquet_row_group_size': 250000, 'output_path': 'data/processed/favorita_merged/favorita_merged_base.parquet', 'storage_approach': 'single Parquet file written incrementally in bounded row groups', 'started_at_utc': '2026-08-07T11:36:44.130926+00:00'}


## 3. Raw paths and output paths

In [2]:
REQUIRED_RAW_FILES = (
    "train.csv",
    "items.csv",
    "stores.csv",
    "transactions.csv",
    "holidays_events.csv",
    "oil.csv",
)

raw_inventory = pd.DataFrame([
    {
        "filename": filename,
        "exists": (RAW_DIR / filename).is_file(),
        "bytes": (RAW_DIR / filename).stat().st_size if (RAW_DIR / filename).is_file() else None,
    }
    for filename in REQUIRED_RAW_FILES
])
display(raw_inventory)

missing_files = raw_inventory.loc[~raw_inventory["exists"], "filename"].tolist()
if missing_files:
    raise FileNotFoundError(f"Missing required raw files: {missing_files}")

,filename,exists,bytes
0,train.csv,True,4997452288
1,items.csv,True,101841
2,stores.csv,True,1387
3,transactions.csv,True,1552637
4,holidays_events.csv,True,22309
5,oil.csv,True,20580


## 4. Load small reference tables

In [3]:
items = pd.read_csv(
    RAW_DIR / "items.csv",
    dtype={"item_nbr": "int32", "family": "string", "class": "int16", "perishable": "int8"},
)
stores = pd.read_csv(
    RAW_DIR / "stores.csv",
    dtype={"store_nbr": "int16", "city": "string", "state": "string", "type": "string", "cluster": "int8"},
).rename(columns={"type": "store_type"})
transactions = pd.read_csv(
    RAW_DIR / "transactions.csv",
    dtype={"store_nbr": "int16", "transactions": "int32"},
    parse_dates=["date"],
)
oil = pd.read_csv(
    RAW_DIR / "oil.csv",
    dtype={"dcoilwtico": "float64"},
    parse_dates=["date"],
)
holidays = pd.read_csv(
    RAW_DIR / "holidays_events.csv",
    dtype={
        "type": "string",
        "locale": "string",
        "locale_name": "string",
        "description": "string",
        "transferred": "boolean",
    },
    parse_dates=["date"],
)

reference_shapes = pd.DataFrame([
    {"source": "items", "rows": len(items), "columns": len(items.columns)},
    {"source": "stores", "rows": len(stores), "columns": len(stores.columns)},
    {"source": "transactions", "rows": len(transactions), "columns": len(transactions.columns)},
    {"source": "oil", "rows": len(oil), "columns": len(oil.columns)},
    {"source": "holidays_events", "rows": len(holidays), "columns": len(holidays.columns)},
])
display(reference_shapes)

,source,rows,columns
0,items,4100,4
1,stores,54,5
2,transactions,83488,3
3,oil,1218,2
4,holidays_events,350,6


## 5. Validate reference tables

In [4]:
reference_key_checks = pd.DataFrame([
    {"source": "items", "key": "item_nbr", "duplicate_key_rows": int(items.duplicated(["item_nbr"]).sum())},
    {"source": "stores", "key": "store_nbr", "duplicate_key_rows": int(stores.duplicated(["store_nbr"]).sum())},
    {"source": "transactions", "key": "date + store_nbr", "duplicate_key_rows": int(transactions.duplicated(["date", "store_nbr"]).sum())},
    {"source": "oil", "key": "date", "duplicate_key_rows": int(oil.duplicated(["date"]).sum())},
])
display(reference_key_checks)

if int(reference_key_checks["duplicate_key_rows"].sum()) != 0:
    raise ValueError("A small reference table violates its expected join-key uniqueness.")
if holidays["transferred"].isna().any():
    raise ValueError("holidays_events.csv contains an unverified missing transferred flag.")

,source,key,duplicate_key_rows
0,items,item_nbr,0
1,stores,store_nbr,0
2,transactions,date + store_nbr,0
3,oil,date,0


## 6. Build the store-date holiday table

National records are expanded to every store. Regional records match `locale_name` to store `state`, and local records match it to store `city`. Multiple applicable records are collapsed into one store-date row. A source record marked `transferred=True` is retained as transfer context but does not by itself set `is_holiday=True`; the destination `Transfer` record supplies the active date.

In [5]:
store_scope = stores[["store_nbr", "city", "state"]].copy()

national_events = holidays.loc[holidays["locale"].eq("National")].assign(_join_key=1).merge(
    store_scope[["store_nbr"]].assign(_join_key=1),
    on="_join_key",
    how="inner",
    validate="many_to_many",
).drop(columns="_join_key")

regional_events = holidays.loc[holidays["locale"].eq("Regional")].merge(
    store_scope[["store_nbr", "state"]],
    left_on="locale_name",
    right_on="state",
    how="inner",
    validate="many_to_many",
).drop(columns="state")

local_events = holidays.loc[holidays["locale"].eq("Local")].merge(
    store_scope[["store_nbr", "city"]],
    left_on="locale_name",
    right_on="city",
    how="inner",
    validate="many_to_many",
).drop(columns="city")

applicable_holidays = pd.concat(
    [national_events, regional_events, local_events],
    ignore_index=True,
)
applicable_holidays["is_holiday_record"] = (
    ~applicable_holidays["transferred"].astype(bool)
    & ~applicable_holidays["type"].eq("Work Day")
)

def combine_distinct_text(values: pd.Series):
    distinct = sorted({str(value) for value in values.dropna() if str(value)})
    return " | ".join(distinct) if distinct else pd.NA

holiday_store_date = applicable_holidays.groupby(["date", "store_nbr"], as_index=False).agg(
    is_holiday=("is_holiday_record", "max"),
    holiday_type=("type", combine_distinct_text),
    holiday_locale=("locale", combine_distinct_text),
    holiday_description=("description", combine_distinct_text),
    holiday_transferred=("transferred", "max"),
    holiday_event_count=("description", "size"),
)
holiday_store_date["is_holiday"] = holiday_store_date["is_holiday"].astype("bool")
holiday_store_date["holiday_transferred"] = holiday_store_date["holiday_transferred"].astype("bool")
holiday_store_date["holiday_event_count"] = holiday_store_date["holiday_event_count"].astype("int16")

holiday_key_duplicates = int(holiday_store_date.duplicated(["date", "store_nbr"]).sum())
if holiday_key_duplicates:
    raise ValueError("Holiday preprocessing did not produce one row per date + store_nbr.")

holiday_validation = pd.DataFrame([{
    "applicable_source_records_after_scope_expansion": len(applicable_holidays),
    "store_date_rows": len(holiday_store_date),
    "duplicate_store_date_rows": holiday_key_duplicates,
    "transferred_context_rows": int(holiday_store_date["holiday_transferred"].sum()),
    "multi_event_store_date_rows": int((holiday_store_date["holiday_event_count"] > 1).sum()),
}])
display(holiday_validation)
display(holiday_store_date.head(10))

,applicable_source_records_after_scope_expansion,store_date_rows,duplicate_store_date_rows,transferred_context_rows,multi_event_store_date_rows
0,9950,9601,0,462,349


,date,store_nbr,is_holiday,holiday_type,holiday_locale,holiday_description,holiday_transferred,holiday_event_count
0,2012-03-02,52,True,Holiday,Local,Fundacion de Manta,False,1
1,2012-03-02,53,True,Holiday,Local,Fundacion de Manta,False,1
2,2012-04-01,12,True,Holiday,Regional,Provincializacion de Cotopaxi,False,1
3,2012-04-01,13,True,Holiday,Regional,Provincializacion de Cotopaxi,False,1
4,2012-04-12,37,True,Holiday,Local,Fundacion de Cuenca,False,1
5,2012-04-12,39,True,Holiday,Local,Fundacion de Cuenca,False,1
6,2012-04-12,42,True,Holiday,Local,Fundacion de Cuenca,False,1
7,2012-04-14,36,True,Holiday,Local,Cantonizacion de Libertad,False,1
8,2012-04-21,14,True,Holiday,Local,Cantonizacion de Riobamba,False,1
9,2012-05-12,22,True,Holiday,Local,Cantonizacion del Puyo,False,1


## 7. Define the chunk merge process

In [6]:
JOIN_SPECS = {
    "items": {"right": items, "keys": ["item_nbr"]},
    "stores": {"right": stores, "keys": ["store_nbr"]},
    "transactions": {"right": transactions, "keys": ["date", "store_nbr"]},
    "oil": {"right": oil, "keys": ["date"]},
    "holidays": {"right": holiday_store_date, "keys": ["date", "store_nbr"]},
}

join_totals = {
    name: {
        "source": name,
        "row_count_before_join": 0,
        "row_count_after_join": 0,
        "unmatched_row_count": 0,
        "duplicate_grain_rows_before": 0,
        "duplicate_grain_rows_after": 0,
    }
    for name in JOIN_SPECS
}

def merge_many_to_one(base: pd.DataFrame, source_name: str) -> pd.DataFrame:
    spec = JOIN_SPECS[source_name]
    indicator = f"_{source_name}_merge"
    before_rows = len(base)
    before_grain_duplicates = int(base.duplicated(GRAIN_COLUMNS).sum())
    merged = base.merge(
        spec["right"],
        on=spec["keys"],
        how="left",
        validate="many_to_one",
        indicator=indicator,
        sort=False,
    )
    after_rows = len(merged)
    after_grain_duplicates = int(merged.duplicated(GRAIN_COLUMNS).sum())
    unmatched = int(merged[indicator].eq("left_only").sum())
    totals = join_totals[source_name]
    totals["row_count_before_join"] += before_rows
    totals["row_count_after_join"] += after_rows
    totals["unmatched_row_count"] += unmatched
    totals["duplicate_grain_rows_before"] += before_grain_duplicates
    totals["duplicate_grain_rows_after"] += after_grain_duplicates
    if after_rows != before_rows:
        raise ValueError(f"{source_name} join changed row count: {before_rows} -> {after_rows}")
    if after_grain_duplicates != before_grain_duplicates:
        raise ValueError(f"{source_name} join introduced duplicate grain rows")
    return merged.drop(columns=indicator)

## 8. Merge train chunks

Each chunk is enriched with the small reference tables and written immediately. Diagnostics are updated incrementally. Source grain ordering is validated so duplicate-grain checks also cover chunk boundaries. Strictly increasing `id` values prove that exact duplicates cannot occur across chunks; exact duplicates within each merged chunk are still counted directly.

In [7]:
if TEMP_OUTPUT_PATH.exists():
    TEMP_OUTPUT_PATH.unlink()

writer = None
output_schema = None
total_rows = 0
chunk_count = 0
null_counts = None
onpromotion_counts = {}
unique_stores = set()
unique_items = set()
minimum_date = None
maximum_date = None
unit_sales_count = 0
unit_sales_sum = 0.0
unit_sales_sum_squares = 0.0
unit_sales_min = math.inf
unit_sales_max = -math.inf
exact_duplicate_rows = 0
grain_duplicate_rows = 0
previous_last_grain = None
previous_last_id = None
source_grain_sorted = True
source_ids_strictly_increasing = True
merge_started = time.perf_counter()

try:
    for chunk in pd.read_csv(
        RAW_DIR / "train.csv",
        dtype={
            "id": "int64",
            "store_nbr": "int16",
            "item_nbr": "int32",
            "unit_sales": "float64",
            "onpromotion": "boolean",
        },
        parse_dates=["date"],
        chunksize=TRAIN_CHUNKSIZE,
    ):
        chunk_count += 1
        source_index = pd.MultiIndex.from_frame(chunk[GRAIN_COLUMNS])
        source_grain_sorted = source_grain_sorted and source_index.is_monotonic_increasing
        current_first_grain = tuple(chunk.iloc[0][GRAIN_COLUMNS])
        current_last_grain = tuple(chunk.iloc[-1][GRAIN_COLUMNS])
        within_chunk_grain_duplicates = int(chunk.duplicated(GRAIN_COLUMNS).sum())
        boundary_grain_duplicate = int(previous_last_grain == current_first_grain) if previous_last_grain is not None else 0
        grain_duplicate_rows += within_chunk_grain_duplicates + boundary_grain_duplicate
        if previous_last_grain is not None and current_first_grain < previous_last_grain:
            source_grain_sorted = False
        previous_last_grain = current_last_grain

        ids = chunk["id"]
        ids_increasing = bool(ids.is_monotonic_increasing and not ids.duplicated().any())
        if previous_last_id is not None and int(ids.iloc[0]) <= previous_last_id:
            ids_increasing = False
        source_ids_strictly_increasing = source_ids_strictly_increasing and ids_increasing
        previous_last_id = int(ids.iloc[-1])

        merged = chunk
        for source_name in JOIN_SPECS:
            merged = merge_many_to_one(merged, source_name)

        merged["is_holiday"] = merged["is_holiday"].fillna(False).astype("bool")
        merged["holiday_transferred"] = merged["holiday_transferred"].fillna(False).astype("bool")
        merged["holiday_event_count"] = merged["holiday_event_count"].fillna(0).astype("int16")

        exact_duplicate_rows += int(merged.duplicated().sum())
        total_rows += len(merged)
        current_nulls = merged.isna().sum().astype("int64")
        null_counts = current_nulls if null_counts is None else null_counts.add(current_nulls, fill_value=0).astype("int64")
        for value, count in merged["onpromotion"].value_counts(dropna=False).items():
            label = "<missing>" if pd.isna(value) else str(bool(value))
            onpromotion_counts[label] = onpromotion_counts.get(label, 0) + int(count)
        unique_stores.update(int(value) for value in merged["store_nbr"].unique())
        unique_items.update(int(value) for value in merged["item_nbr"].unique())
        chunk_min_date = merged["date"].min()
        chunk_max_date = merged["date"].max()
        minimum_date = chunk_min_date if minimum_date is None else min(minimum_date, chunk_min_date)
        maximum_date = chunk_max_date if maximum_date is None else max(maximum_date, chunk_max_date)
        sales = merged["unit_sales"].dropna().astype("float64")
        unit_sales_count += len(sales)
        unit_sales_sum += float(sales.sum())
        unit_sales_sum_squares += float(np.square(sales.to_numpy()).sum())
        unit_sales_min = min(unit_sales_min, float(sales.min()))
        unit_sales_max = max(unit_sales_max, float(sales.max()))

        table = pa.Table.from_pandas(merged, preserve_index=False)
        if writer is None:
            output_schema = table.schema
            writer = pq.ParquetWriter(
                TEMP_OUTPUT_PATH,
                output_schema,
                compression=PARQUET_COMPRESSION,
            )
        elif table.schema != output_schema:
            table = table.cast(output_schema, safe=True)
        writer.write_table(table, row_group_size=PARQUET_ROW_GROUP_SIZE)

        if chunk_count % 25 == 0:
            print(f"Processed {total_rows:,} rows across {chunk_count} chunks", flush=True)

    if writer is None:
        raise ValueError("train.csv produced no chunks")
    writer.close()
    writer = None
    if not source_grain_sorted:
        raise ValueError("Source grain is not globally sorted; boundary-safe duplicate counting cannot be certified.")
    if not source_ids_strictly_increasing:
        raise ValueError("Source id values are not globally unique and strictly increasing.")
    TEMP_OUTPUT_PATH.replace(OUTPUT_PATH)
except Exception:
    if writer is not None:
        writer.close()
    if TEMP_OUTPUT_PATH.exists():
        TEMP_OUTPUT_PATH.unlink()
    raise

merge_seconds = time.perf_counter() - merge_started
print({
    "chunks_processed": chunk_count,
    "final_rows_written": total_rows,
    "merge_seconds": round(merge_seconds, 3),
    "output_bytes": OUTPUT_PATH.stat().st_size,
})

Processed 25,000,000 rows across 25 chunks


Processed 50,000,000 rows across 50 chunks


Processed 75,000,000 rows across 75 chunks


Processed 100,000,000 rows across 100 chunks


Processed 125,000,000 rows across 125 chunks


{'chunks_processed': 126, 'final_rows_written': 125497040, 'merge_seconds': 219.442, 'output_bytes': 723900039}


## 9. Write merged Parquet

In [8]:
parquet_file = pq.ParquetFile(OUTPUT_PATH)
parquet_metadata = pd.DataFrame([{
    "path": OUTPUT_PATH.relative_to(REPO_ROOT).as_posix(),
    "storage_approach": "single Parquet file with incremental row groups",
    "compression": PARQUET_COMPRESSION,
    "file_bytes": OUTPUT_PATH.stat().st_size,
    "metadata_rows": parquet_file.metadata.num_rows,
    "metadata_columns": parquet_file.metadata.num_columns,
    "row_groups": parquet_file.metadata.num_row_groups,
}])
display(parquet_metadata)

if parquet_file.metadata.num_rows != total_rows:
    raise ValueError("Saved Parquet metadata row count does not match rows written.")

,path,storage_approach,compression,file_bytes,metadata_rows,metadata_columns,row_groups
0,data/processed/favorita_merged/favorita_merged...,single Parquet file with incremental row groups,zstd,723900039,125497040,21,502


## 10. Validate final dataset

In [9]:
join_validation = pd.DataFrame(join_totals.values())
join_validation["row_count_changed"] = (
    join_validation["row_count_before_join"] != join_validation["row_count_after_join"]
)
join_validation["duplicate_grain_rows_introduced"] = (
    join_validation["duplicate_grain_rows_after"] - join_validation["duplicate_grain_rows_before"]
)
join_validation["matched_row_count"] = (
    join_validation["row_count_after_join"] - join_validation["unmatched_row_count"]
)
join_validation["matched_percentage"] = (
    100.0 * join_validation["matched_row_count"] / join_validation["row_count_after_join"]
)
display(join_validation[[
    "source",
    "row_count_before_join",
    "row_count_after_join",
    "row_count_changed",
    "unmatched_row_count",
    "matched_row_count",
    "matched_percentage",
    "duplicate_grain_rows_introduced",
]])

if join_validation["row_count_changed"].any():
    raise ValueError("At least one join changed the training row count.")
if (join_validation["duplicate_grain_rows_introduced"] != 0).any():
    raise ValueError("At least one join introduced duplicate grain rows.")

final_overview = pd.DataFrame([{
    "final_row_count": total_rows,
    "final_column_count": len(output_schema),
    "exact_duplicate_row_count": exact_duplicate_rows,
    "duplicate_date_store_item_count": grain_duplicate_rows,
    "unique_store_count": len(unique_stores),
    "unique_item_count": len(unique_items),
    "minimum_date": minimum_date.date().isoformat(),
    "maximum_date": maximum_date.date().isoformat(),
    "source_grain_sorted": source_grain_sorted,
    "source_ids_strictly_increasing": source_ids_strictly_increasing,
}])
display(final_overview)
print("Full column list:")
print(output_schema.names)

,source,row_count_before_join,row_count_after_join,row_count_changed,unmatched_row_count,matched_row_count,matched_percentage,duplicate_grain_rows_introduced
0,items,125497040,125497040,False,0,125497040,100.000000,0
1,stores,125497040,125497040,False,0,125497040,100.000000,0
2,transactions,125497040,125497040,False,214625,125282415,99.828980,0
3,oil,125497040,125497040,False,37786672,87710368,69.890388,0
4,holidays,125497040,125497040,False,113841496,11655544,9.287505,0


,final_row_count,final_column_count,exact_duplicate_row_count,duplicate_date_store_item_count,unique_store_count,unique_item_count,minimum_date,maximum_date,source_grain_sorted,source_ids_strictly_increasing
0,125497040,21,0,0,54,4036,2013-01-01,2017-08-15,True,True


Full column list:
['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion', 'family', 'class', 'perishable', 'city', 'state', 'store_type', 'cluster', 'transactions', 'dcoilwtico', 'is_holiday', 'holiday_type', 'holiday_locale', 'holiday_description', 'holiday_transferred', 'holiday_event_count']


## 11. Display the first 10 rows

In [10]:
first_row_group = parquet_file.read_row_group(0)
final_sample = first_row_group.slice(0, 10).to_pandas()
display(final_sample.head(10))
print(f"Bounded saved-Parquet read succeeded: {len(final_sample.head(10))} rows displayed.")

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,...,store_type,cluster,transactions,dcoilwtico,is_holiday,holiday_type,holiday_locale,holiday_description,holiday_transferred,holiday_event_count
0,0,2013-01-01,25,103665,7.0,<NA>,BREAD/BAKERY,2712,1,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
1,1,2013-01-01,25,105574,1.0,<NA>,GROCERY I,1045,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
2,2,2013-01-01,25,105575,2.0,<NA>,GROCERY I,1045,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
3,3,2013-01-01,25,108079,1.0,<NA>,GROCERY I,1030,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
4,4,2013-01-01,25,108701,1.0,<NA>,DELI,2644,1,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
5,5,2013-01-01,25,108786,3.0,<NA>,CLEANING,3044,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
6,6,2013-01-01,25,108797,1.0,<NA>,GROCERY I,1004,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
7,7,2013-01-01,25,108952,1.0,<NA>,CLEANING,3024,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
8,8,2013-01-01,25,111397,13.0,<NA>,GROCERY I,1072,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
9,9,2013-01-01,25,114790,3.0,<NA>,GROCERY I,1004,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1


Bounded saved-Parquet read succeeded: 10 rows displayed.


## 12. Null and duplicate diagnostics

In [11]:
data_types = pd.DataFrame([
    {"column": field.name, "parquet_dtype": str(field.type)}
    for field in output_schema
])
display(data_types)

null_diagnostics = null_counts.rename_axis("column").reset_index(name="null_count")
null_diagnostics["null_percentage"] = 100.0 * null_diagnostics["null_count"] / total_rows
display(null_diagnostics)

unit_sales_mean = unit_sales_sum / unit_sales_count
unit_sales_variance = (
    (unit_sales_sum_squares - (unit_sales_sum ** 2) / unit_sales_count) / (unit_sales_count - 1)
    if unit_sales_count > 1 else math.nan
)
unit_sales_summary = pd.DataFrame([{
    "count": unit_sales_count,
    "mean": unit_sales_mean,
    "std": math.sqrt(max(0.0, unit_sales_variance)),
    "min": unit_sales_min,
    "max": unit_sales_max,
}])
display(unit_sales_summary)

onpromotion_summary = pd.DataFrame(
    [{"onpromotion": label, "row_count": count} for label, count in sorted(onpromotion_counts.items())]
)
display(onpromotion_summary)

,column,parquet_dtype
0,id,int64
1,date,timestamp[us]
2,store_nbr,int16
3,item_nbr,int32
4,unit_sales,double
5,onpromotion,bool
6,family,large_string
7,class,int16
8,perishable,int8
9,city,large_string


,column,null_count,null_percentage
0,id,0,0.000000
1,date,0,0.000000
2,store_nbr,0,0.000000
3,item_nbr,0,0.000000
4,unit_sales,0,0.000000
5,onpromotion,21657651,17.257499
6,family,0,0.000000
7,class,0,0.000000
8,perishable,0,0.000000
9,city,0,0.000000


,count,mean,std,min,max
0,125497040,8.554865,23.605152,-15372.0,89440.0


,onpromotion,row_count
0,<missing>,21657651
1,False,96028767
2,True,7810622


## 13. Join coverage diagnostics

In [12]:
join_coverage = join_validation[[
    "source",
    "matched_row_count",
    "unmatched_row_count",
    "matched_percentage",
]].copy()
display(join_coverage)

,source,matched_row_count,unmatched_row_count,matched_percentage
0,items,125497040,0,100.000000
1,stores,125497040,0,100.000000
2,transactions,125282415,214625,99.828980
3,oil,87710368,37786672,69.890388
4,holidays,11655544,113841496,9.287505


## 14. Final summary and next-step notes

- The saved artifact is an **uncleaned merged base dataset** built from real Favorita raw files.
- Missing values have not yet been imputed. The only filled contextual values are `is_holiday=False`, `holiday_transferred=False`, and `holiday_event_count=0` where no store-date holiday record applies.
- Negative `unit_sales` values have not been removed, clipped, or otherwise modified.
- No feature engineering has been performed: no lags, rolling statistics, encodings, scaling, or derived forecasting features were added.
- No model-training transformations, train/validation splits, model fitting, prediction generation, or Kaggle submission logic were performed.
- Cleaning decisions will be made only after reviewing this merged dataset and its visible diagnostics.